# SNNAI v3.5 Small Modular LM Training

This notebook trains the SNNAI small language model on a tiny corpus using a T4 GPU.

In [ ]:
!pip install -q snntorch==0.9.4 torch torchvision numpy

In [ ]:
import os
import sys
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate
import urllib.request

# Clone the SNNAI repository
REPO_URL = "https://github.com/sabunosuke1008-create/snnai.git"
BRANCH = "v3.5"
if not os.path.exists("/kaggle/working/snnai"):
    !git clone --depth 1 --branch $BRANCH $REPO_URL /kaggle/working/snnai
sys.path.insert(0, "/kaggle/working/snnai")

In [ ]:
from snnai.modules.language import CharTokenizer, SpikeEncoder
from snnai.modules.language.next_token import NextTokenSNN

# Tiny corpus for proof-of-concept
corpus = "hello world hello snnai event driven neural network low power ai " * 50
vocab = "".join(sorted(set(corpus)))
tokenizer = CharTokenizer(vocab)
encoder = SpikeEncoder(vocab_size=tokenizer.vocab_size, time_steps=20)
model = NextTokenSNN(vocab_size=tokenizer.vocab_size, hidden_size=128)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Prepare dataset
seq_len = 20
data = tokenizer.encode(corpus[:1000])
inputs, targets = [], []
for i in range(len(data) - seq_len):
    inputs.append(data[i:i+seq_len])
    targets.append(data[i+1:i+seq_len+1])
inputs = torch.tensor(inputs[:256]).to(device)
targets = torch.tensor(targets[:256]).to(device)

In [ ]:
# Training loop
model.train()
for epoch in range(50):
    total_loss = 0.0
    for b in range(0, len(inputs), 32):
        x = inputs[b:b+32]
        y = targets[b:b+32]
        spikes = encoder(x).to(device)
        out = model(spikes)
        loss = criterion(out.view(-1, tokenizer.vocab_size), y.view(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
        print(f"Epoch {epoch}, loss {total_loss:.4f}")
print("Training complete")

In [ ]:
# Generation demo
model.eval()
prompt = "hello "
text = prompt
for _ in range(20):
    indices = torch.tensor([tokenizer.encode(text[-seq_len:])]).to(device)
    spikes = encoder(indices)
    out = model(spikes)
    next_idx = int(torch.argmax(out[0, -1, :]).item())
    text += tokenizer.idx_to_char[next_idx]
print("Generated:", text)